# LatentDriver Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_colab_runner.ipynb)

Use this notebook as a terminal-style launcher. Notebook cells are shell commands; clone/pull, Drive binding, and execution logic live in `scripts/*.py` and `src/`.


## Operating Model

1. Run the Drive mount code cell. This is a Colab platform exception, not experiment logic.
2. Run the GCS auth code cell if you want to evaluate directly from `gs://waymo_open_dataset_motion_v_1_1_0`. Skip it only if you intentionally use the Drive-local `assets/raw_womd` cache.
3. Run the bootstrap shell cell.
4. Optionally list profiles or check preprocessing markers.
5. Run the full reactive single-model gate cell. This is the next milestone after successful full preprocessing and dry-run readiness.
6. Inspect the debug status shell cell if a profile fails.

Only the first two code cells are notebook-native Python. Every other cell stays shell-first so the runnable logic remains in `scripts/` and `src/`.


## Mount Google Drive

Mount Drive once per Colab runtime so checkpoints, preprocessed artifacts, run bundles, and debug outputs can bind to the persistent project folder.


In [1]:
from pathlib import Path

DRIVE_MOUNTPOINT = "/content/drive"
drive_root = Path(DRIVE_MOUNTPOINT) / "MyDrive"

if drive_root.is_dir():
    print(f"Drive already mounted at {drive_root}")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNTPOINT)
    print(f"Mounted Drive at {drive_root}")


Mounted at /content/drive
Mounted Drive at /content/drive/MyDrive


## Authenticate GCS Access

Authenticate the Colab runtime for direct `gs://waymo_open_dataset_motion_v_1_1_0` reads. Keep this enabled unless you intentionally switch to the Drive-local raw WOMD cache.


In [2]:
from __future__ import annotations

import json
import subprocess

USE_GCS_DIRECT = True

if USE_GCS_DIRECT:
    from google.colab import auth
    auth.authenticate_user()
    checks = {}
    commands = {
        "active_account": ["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=json"],
        "adc_token": ["gcloud", "auth", "application-default", "print-access-token"],
    }
    for name, command in commands.items():
        proc = subprocess.run(command, text=True, capture_output=True, check=False)
        payload = {"command": command, "returncode": proc.returncode}
        if name == "adc_token":
            payload["token_ready"] = proc.returncode == 0 and bool(proc.stdout.strip())
            payload["stderr"] = proc.stderr.strip()
        else:
            payload["stdout"] = proc.stdout.strip()
            payload["stderr"] = proc.stderr.strip()
        checks[name] = payload
    print(json.dumps({"use_gcs_direct": True, "checks": checks}, indent=2))
else:
    print(json.dumps({
        "use_gcs_direct": False,
        "message": "Skipping GCS auth because this session will use the Drive-local assets/raw_womd cache.",
    }, indent=2))


{
  "use_gcs_direct": true,
  "checks": {
    "active_account": {
      "command": [
        "gcloud",
        "auth",
        "list",
        "--filter=status:ACTIVE",
        "--format=json"
      ],
      "returncode": 0,
      "stdout": "[\n  {\n    \"account\": \"the.achyut.morang@gmail.com\",\n    \"status\": \"ACTIVE\"\n  }\n]",
      "stderr": ""
    },
    "adc_token": {
      "command": [
        "gcloud",
        "auth",
        "application-default",
        "print-access-token"
      ],
      "returncode": 0,
      "token_ready": true,
      "stderr": ""
    }
  }
}


## Bootstrap Repository And Bind Artifacts

Clone or fast-forward `main`, validate the Drive mount, and bind the repository artifact paths to persistent Drive-backed storage.


In [3]:
%%bash
set -euo pipefail

BOOTSTRAP=/tmp/latentdriver_colab_bootstrap.py
curl -fsSL \
  https://raw.githubusercontent.com/achyutmorang/latentdriver-waymax-experiments/main/scripts/colab_bootstrap.py \
  -o "$BOOTSTRAP"

python3 "$BOOTSTRAP" \
  --repo-url https://github.com/achyutmorang/latentdriver-waymax-experiments.git \
  --branch main \
  --repo-dir /content/latentdriver-waymax-experiments \
  --drive-base-root /content/drive/MyDrive/waymax_research \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


[colab-bootstrap] $ git clone --branch main https://github.com/achyutmorang/latentdriver-waymax-experiments.git /content/latentdriver-waymax-experiments
{
  "drive_binding": {
    "checkpoints": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/checkpoints",
    "debug_runs": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/debug_runs",
    "drive_root": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments",
    "preprocessed": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/preprocessed",
    "raw_womd": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/raw_womd",
    "results": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/results/runs",
    "smoke": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/smoke"
  },
  "drive_mount": {
    "mode": "already_mounted",
    "mounted": true,
    "mountpoint": "/

Cloning into '/content/latentdriver-waymax-experiments'...


## List Available Runner Profiles

Print the supported `colab_canary.py` profiles. Use this when you need to confirm the exact profile names before launching a run.


In [4]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
python3 scripts/colab_canary.py --list-profiles


{
  "bootstrap-session": "Install runtime, verify checkpoints, and run the full_reactive dry-run preflight.",
  "download-checkpoints": "Download or verify public evaluation checkpoints into the Drive-bound checkpoint cache.",
  "env-check": "Capture environment, git, GPU, and artifact status without running heavy commands.",
  "full-eval-dry-run": "Dry-run full_reactive evaluation for one model without launching simulation.",
  "full-eval-non-reactive": "Run all public checkpoints on full_non_reactive.",
  "full-eval-non-reactive-single": "Run one model on full_non_reactive.",
  "full-eval-reactive": "Run all public checkpoints on full_reactive.",
  "full-eval-reactive-single": "Run one model on full_reactive.",
  "full-preprocess": "Run full validation preprocessing; use only when you intentionally want to rebuild/resume preprocessing.",
  "full-preprocess-status": "Check full preprocess artifact paths without scanning the large cache directories.",
  "install-runtime": "Install/patc

## Confirm Git Revision

Fast-forward the local Colab checkout and print the active commit. This is a lightweight sanity check before expensive runs.


In [11]:
%%bash
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main
git rev-parse --short HEAD


Updating be9f3d9..01232e2
Fast-forward
 scripts/preprocess_validation_only.py    | 72 ++++++++++++++++++++++++++------
 tests/test_preprocess_validation_only.py | 28 +++++++++++++
 2 files changed, 87 insertions(+), 13 deletions(-)
01232e2


From https://github.com/achyutmorang/latentdriver-waymax-experiments
 * branch            main       -> FETCH_HEAD
   be9f3d9..01232e2  main       -> origin/main


## Full Evaluation Dry-Run Gate

Run the fast readiness gate for `full_reactive` without launching simulation. This should report `ready: true` before the real full evaluation cell is used.


In [13]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

# Optional readiness check. This should stay fast and should report ready: true before real full evaluation.
python3 scripts/colab_canary.py \
  --profile full-eval-dry-run \
  --model latentdriver_t2_j3 \
  --seed 0 \
  --vis false \
  --auto-install-runtime \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


Already up to date.

[canary] step 1: bootstrap_upstream
[canary] $ /usr/bin/python3 scripts/bootstrap_upstream.py
M	configs/simulate.yaml
M	simulate.py
M	simulator/engines/base_simulator.py
M	simulator/engines/ltd_simulator.py
M	simulator/metric.py
M	simulator/utils.py
M	src/policy/baseline/bc_baseline.py
M	src/policy/latentdriver/lantentdriver_model.py
M	src/utils/utils.py
M	waymax/agents/expert.py
M	waymax/agents/waypoint_following_agent.py
M	waymax/visualization/utils.py
M	waymax/visualization/viz.py
{
  "fork_repo_url": "https://github.com/achyutmorang/LatentDriver.git",
  "patch_path": "/content/latentdriver-waymax-experiments/patches/latentdriver_eval_contract.patch",
  "patch_state": "already_applied",
  "pinned_commit": "721bd6e1f87373457b743d0e0a9606d4d0727b6f",
  "repo_dir": "/content/latentdriver-waymax-experiments/external/LatentDriver",
  "upstream_repo_url": "https://github.com/Sephirex-X/LatentDriver.git"
}
[canary] step 1 returncode=0 duration=0.4s

[canary] step 2: se

From https://github.com/achyutmorang/latentdriver-waymax-experiments
 * branch            main       -> FETCH_HEAD
HEAD is now at 721bd6e Update README.md
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Running command git clone --filter=blob:none --quiet https://github.com/waymo-research/waymax.git /tmp/pip-install-d56ga09b/waymo-waymax_fba75b2408ba4b3a85587cc3be487b6d
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confl

## Check Full Preprocess Markers

Verify that full preprocessing wrote `_SUCCESS` and `preprocess_manifest.json`. These markers prevent accidental full eval runs on partial preprocessing caches.


In [15]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

PREPROCESS_DIR=/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path
test -f "$PREPROCESS_DIR/_SUCCESS" && echo "_SUCCESS present" || echo "_SUCCESS missing"
test -f "$PREPROCESS_DIR/preprocess_manifest.json" && echo "manifest present" || echo "manifest missing"


_SUCCESS present
manifest present


## Run First Full Reactive Simulation

Launch the next milestone: one resumable full reactive simulation/evaluation run for `latentdriver_t2_j3`. If Colab disconnects, rerun this same cell to skip completed shards and continue.


In [17]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

# Next milestone: one resumable full reactive simulation/evaluation run for LatentDriver T2-J3.
# If Colab disconnects, rerun this same cell; completed shards are skipped.
python3 scripts/colab_canary.py \
  --profile full-eval-reactive-single \
  --model latentdriver_t2_j3 \
  --seed 0 \
  --vis false \
  --auto-install-runtime \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0



[canary] step 1: bootstrap_upstream
[canary] $ /usr/bin/python3 scripts/bootstrap_upstream.py
M	configs/simulate.yaml
M	simulate.py
M	simulator/engines/base_simulator.py
M	simulator/engines/ltd_simulator.py
M	simulator/metric.py
M	simulator/utils.py
M	src/ops/sort_vertices/sort_vert.py
M	src/policy/baseline/bc_baseline.py
M	src/policy/latentdriver/gpt2_model.py
M	src/policy/latentdriver/lantentdriver_model.py
M	src/preprocess/preprocess_data.py
M	src/utils/utils.py
M	waymax/agents/expert.py
M	waymax/agents/waypoint_following_agent.py
M	waymax/visualization/utils.py
M	waymax/visualization/viz.py
{
  "fork_repo_url": "https://github.com/achyutmorang/LatentDriver.git",
  "patch_path": "/content/latentdriver-waymax-experiments/patches/latentdriver_eval_contract.patch",
  "patch_state": "already_applied",
  "pinned_commit": "721bd6e1f87373457b743d0e0a9606d4d0727b6f",
  "repo_dir": "/content/latentdriver-waymax-experiments/external/LatentDriver",
  "upstream_repo_url": "https://github.com/S

HEAD is now at 721bd6e Update README.md


## Inspect Latest Debug Bundle

Use this after a failed canary run. It prints the latest run manifest and latest failure summary without requiring manual traceback hunting.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

# Use this after any failed canary run.
python3 - <<'PY'
import json
from pathlib import Path

latest = Path("results/debug_runs/latest/manifest.json")
latest_failure = Path("results/debug_runs/latest_failure/failure_summary.json")
for path in (latest, latest_failure):
    print(f"\n== {path} ==")
    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2)[:12000])
    else:
        print("missing")
PY
